## Personal Productivity Agent with Tool Calling

### Tools to Build

You must define **4 Python functions** as tools. Each function must have a clear docstring — AISuite uses it to describe the tool to the LLM.

| Tool                     | What it does                                                                      | Parameters                                 |
| ------------------------ | --------------------------------------------------------------------------------- | ------------------------------------------ |
| `get_current_datetime` | Returns the current date and time as a formatted string                           | None                                       |
| `get_weather_summary`  | Detects location from IP and returns current temp, daily high, and low in Celsius | None                                       |
| `manage_todo_list`     | Reads or appends tasks to a local `todo.txt` file                               | `action` (read/add), `task` (optional) |
| `generate_meeting_qr`  | Generates a QR code PNG from a meeting URL                                        | `url`, `filenam`                       |

#### So let's build this tools one by one

### Tool - 1

In [17]:
from datetime import datetime

def get_current_time():
    """
    Returns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")
get_current_time()

'14:34:58'

### Tool - 2

In [18]:
import requests

def get_weather_summary():
    """ Detects users location from IP and returns current temp, daily high, and low in Celsius.
    """

    # Get location coordinates from the IP address
    lat, lon = requests.get('https://ipinfo.io/json').json()["loc"].split(",")

    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "celsius",
        "timezone": "auto"
    }

    # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    current = weather_data['current']['temperature_2m']
    high = weather_data['daily']['temperature_2m_max'][0]
    low = weather_data['daily']['temperature_2m_min'][0]

    return f"Current: {current}°C, High: {high}°C, Low: {low}°C"

get_weather_summary()

'Current: 33.8°C, High: 34.0°C, Low: 26.7°C'

### Tool - 3

In [19]:
import json
from pathlib import Path
import os

FILE_PATH = Path("tasks.json")

FILE_PATH.parent.mkdir(parents=True, exist_ok=True)

def load_tasks():
    if FILE_PATH.exists():
        try:
            with FILE_PATH.open("r", encoding="utf-8") as f:
                return json.load(f)
        except (json.JSONDecodeError, FileNotFoundError):
            return []
    return []

def save_tasks(tasks):
    with open(FILE_PATH, "w") as f:
        json.dump(tasks, f, indent=4)

def add_task(task: str):
    """
    Adds a new task to the user's to-do list and saves it to tasks.json.
    task: a string describing the task to add, e.g. 'Call John at 5PM' or 'Buy groceries'.
    """
    tasks = load_tasks()
    tasks.append(task)
    save_tasks(tasks)
    return f"Task added: {task}"

def show_tasks():
    """
    Returns all tasks currently in the to-do list as a numbered string.
    """
    tasks = load_tasks()
    if not tasks:
        return "No tasks available."
    return "\n".join(f"{i}. {t}" for i, t in enumerate(tasks, 1))

def delete_task(index: int):
    """
    Deletes a task by its 1-based position number from the to-do list.
    index: the task number as shown by show_tasks(), e.g. 1 for the first task, 2 for the second.
    """
    tasks = load_tasks()

    if 0 <= index <= len(tasks):
        removed = tasks.pop(index-1)
        save_tasks(tasks)
        return f"Task '{index}: {removed}' deleted."
    return f"Invalid index {index}. There are {len(tasks)} tasks."

In [20]:
# Quick test
print(add_task("Leave for the gym 6 am"))
print(show_tasks())
print(delete_task(1))
print(show_tasks())

Task added: Leave for the gym 6 am
1. Leave for the gym 6 am
2. Leave for the gym 6 am
Task '1: Leave for the gym 6 am' deleted.
1. Leave for the gym 6 am


### Tool - 4

In [21]:
from qrcode.image.styledpil import StyledPilImage
import qrcode

# Create a QR code
def generate_qr_code(data: str, filename: str, image_path: str = None):
    """Generate a QR code image given data and an image path.

    Args:
        data: Text or URL to encode
        filename: Name for the output PNG file (without extension)
        image_path: Path to the image to be used in the QR code
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)

    img = qr.make_image(image_factory=StyledPilImage)
    output_file = f"{filename}.png"
    img.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."

In [22]:
generate_qr_code("http://localhost:3128/oauth/callback?login_option=google&code=2c99a7e2-8595-49ea-bce6-fa2d6fc74fab&state=b1a19649-a19f-4a90-9d64-60aad43c779f", "localhost")

'QR code saved as localhost.png containing: http://localhost:3128/oauth/callback?login_option=...'

#### Now we have all tools/function defined so let's build workflow which can use this tools in natural langual

In [23]:
import json
# import display_functions
from dotenv import load_dotenv
_ = load_dotenv()

In [24]:
import aisuite as ai

# Create an instance of the AISuite client
client = ai.Client()

#### Now Let's define tools manually 

In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns the current time as a string.",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_summary",
            "description": "Detects the user's location from their IP and returns the current temperature, daily high, and low in Celsius.",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "show_tasks",
            "description": "Returns all tasks currently in the to-do list as a numbered string.",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_task",
            "description": "Adds a new task to the user's to-do list and saves it to tasks.json.",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {
                        "type": "string",
                        "description": "A string describing the task to add, e.g. 'Call John at 5PM' or 'Buy groceries'."
                    }
                },
                "required": ["task"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "delete_task",
            "description": "Deletes a task by its 1-based position number from the to-do list. Call show_tasks first to get the task numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {
                        "type": "integer",
                        "description": "The 1-based task number to delete, e.g. 1 for the first task, 2 for the second."
                    }
                },
                "required": ["index"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_qr_code",
            "description": "Generates a QR code PNG image from a URL or text and saves it locally.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "string",
                        "description": "The URL or text to encode in the QR code."
                    },
                    "filename": {
                        "type": "string",
                        "description": "Name for the output PNG file without the extension. Infer a short name from the URL or context if the user did not specify one, e.g. 'gemini_qr' for https://gemini.google.com."
                    },
                    "image_path": {
                        "type": "string",
                        "description": "Optional path to an image to embed in the center of the QR code."
                    }
                },
                "required": ["data", "filename"]
            }
        }
    }
]

In [26]:
# prompt = "can you generate qr code for  this url? : https://gemini.google.com/app"
def generate_response(prompt: str, model: str = "openai:gpt-4o"):
    response = client.chat.completions.create(
        model=model,
        messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
        tools=tools, # <-- Your list of tools with get_current_time
        # max_turns=5 # <-- When defining tools manually, you must handle calls yourself and cannot use max_turns
    )
    return response

In [34]:
## let's format the response in readable format 
print(json.dumps(generate_response("Add 'Call dentist at 3PM' to my to-do list").model_dump(), indent=2, default=str))

{
  "id": "chatcmpl-DsPY60UqhFOxQbBPEoyqstmzQCgSw",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_cpWzRiXScrovx3Kbr6HcDAap",
            "function": {
              "arguments": "{\"task\":\"Call dentist at 3PM\"}",
              "name": "add_task"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1781860294,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": "fp_aa4b0e4fd9",
  "usage": {
    "completion_tokens": 19,
    "prompt_tokens": 341,
    "total_tokens": 360,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,

Now we have reponse with just tool defination here Model unable to run this tool for us, we need run defined tool for him and return reponse of that tool as message with second call

In [28]:
def return_tool_response(response, messages, model: str = "openai:gpt-4o"):

    response2 = None
    tool_result = None

    # Map tool names to their functions
    tool_registry = {
        "get_current_time": get_current_time,
        "get_weather_summary": get_weather_summary,
        "show_tasks": show_tasks,
        "add_task": add_task,
        "delete_task": delete_task,
        "generate_qr_code": generate_qr_code,
    }

    if response.choices[0].message.tool_calls:
        tool_call = response.choices[0].message.tool_calls[0]
        tool_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        # Look up and call the function with the arguments the LLM provided
        tool_result = tool_registry[tool_name](**args)


        messages.append(response.choices[0].message)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

        response2 = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
        )

        return response2.choices[0].message.content

You have now implemented your own manual handling of LLM tool calls. You can choose to have the tools automatically given to your LLM with `max_turns` or write the schema and handle the intermediate parts manually.

## Here is final result

In [ ]:
from IPython.display import display, HTML
import json

def personal_agent(prompt: str, model: str = "openai:gpt-4o", max_turns: int = 5):
    messages = [{"role": "user", "content": prompt}]

    tool_registry = {
        "get_current_time": get_current_time,
        "get_weather_summary": get_weather_summary,
        "show_tasks": show_tasks,
        "add_task": add_task,
        "delete_task": delete_task,
        "generate_qr_code": generate_qr_code,
    }

    html = f"""
    <div style="border-left:4px solid #9b59b6; padding:10px; margin:10px 0; background:#f5eeff;">
        <strong>💬 User Prompt:</strong>
        <p style="color:#000; margin:4px 0;">{prompt}</p>
    </div>
    """

    tool_sequence = []
    final = None
    turns = 0

    while turns < max_turns:
        response = client.chat.completions.create(
            model=model, messages=messages, tools=tools
        )
        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            turns += 1
            tool_call = choice.message.tool_calls[0]
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            tool_sequence.append(tool_name)

            html += f"""
            <div style="border-left:4px solid #444; padding:10px; margin:10px 0; background:#f0f0f0;">
                <strong>🧠 LLM decided to call [turn {turns}/{max_turns}]:</strong> <code>{tool_name}</code>
                <pre style="color:#000; font-size:13px; margin:6px 0;">{json.dumps(args, indent=2)}</pre>
            </div>
            """

            tool_result = tool_registry[tool_name](**args)

            html += f"""
            <div style="border-left:4px solid #007bff; padding:10px; margin:10px 0; background:#eef6ff;">
                <strong>🔧 Tool Result:</strong> <code>{tool_name}</code>
                <pre style="color:#000; font-size:13px; margin:6px 0;">{tool_result}</pre>
            </div>
            """

            messages.append(choice.message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_result)
            })

        else:
            final = choice.message.content
            break
    else:
        # max_turns reached without a stop
        final = f"⚠️ Stopped after {max_turns} tool calls without a final answer."

    html += f"""
    <div style="border-left:4px solid #28a745; padding:10px; margin:10px 0; background:#eafbe7;">
        <strong>✅ Final Answer:</strong>
        <p style="color:#000; margin:4px 0;">{final}</p>
    </div>
    """

    if tool_sequence:
        sequence_str = " → ".join(f"🔧 {t}" for t in tool_sequence)
        html += f"""
        <div style="border-left:4px solid #666; padding:10px; margin:10px 0; background:#f8f9fa;">
            <strong>🧭 Tool Sequence:</strong>
            <p style="color:#000; margin:4px 0;">💬 User → 🧠 LLM → {sequence_str} → ✅ Answer</p>
        </div>
        """

    display(HTML(html))

personal_agent("delete task number 2")

### Debug hint

--> The three things that can go wrong at this stage:

1. Wrong tool picked — your tool descriptions are too similar or ambiguous   
2. Wrong argument value — your parameter description didn't constrain the LLM enough      
3. Wrong argument type — your parameter schema has wrong type (e.g. string instead of integer for index)      